# Session 07, Notebook 2: dataset, evaluators, experiment, regression

**Goal.** Score two prompt versions of the Research Brief Agent on the same cases with Langfuse, read the results per prompt version,
then rerun the cases of fixed bugs (regression).

Order, as on the slides: cases → **dataset** → prompts from Langfuse → **task** → **evaluators** (code and LLM judge) → **experiment**
v1 vs v2 → scores per prompt version → **regression**.

Before you start: the prompts `research-brief` (labels `v1`, `v2`) and `research-brief-judge` exist in Langfuse, copied from the files in `prompts/` (README, step 5).
Docs: [Datasets](https://langfuse.com/docs/evaluation/experiments/datasets) ·
[Experiments via SDK](https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk) ·
[Scores](https://langfuse.com/docs/evaluation/scores/overview) ·
[Link prompts to traces](https://langfuse.com/docs/prompt-management/features/link-to-traces)

In [1]:
import logging, os, sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
from langfuse import get_client

load_dotenv("../.env")
sys.path.insert(0, "..")                     # the session folder, for app/tools.py
for key in ("OPENAI_API_KEY", "TAVILY_API_KEY", "MODEL", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"):
    if not os.getenv(key):
        raise SystemExit(f"{key} is missing: copy .env.example to .env and fill it in (README, step 3)")
langfuse = get_client()
if not langfuse.auth_check():
    raise SystemExit("Langfuse rejected the keys: check LANGFUSE_* in .env and that LANGFUSE_BASE_URL is your region")
logger.info("Langfuse connected | model: %s", os.getenv("MODEL"))

INFO | Langfuse connected | model: openai:gpt-6-luna


## 1. The eval cases
One case per line in `data/eval_cases.jsonl`. `source` says where it came from: `requirement` (a product rule) or `failure` (something that
went wrong). `expected` holds only the keys an evaluator needs. Tags select groups: `smoke` (12 quick cases), `regression` (fixed bugs).

In [2]:
import json
from collections import Counter

cases = [json.loads(line) for line in open("../data/eval_cases.jsonl") if line.strip()]
print(len(cases), "cases", dict(Counter(c["source"] for c in cases)))
print(json.dumps(cases[0]))

26 cases {'requirement': 20, 'failure': 6}
{"id": "R1-01", "input": {"question": "How does a LangGraph checkpointer save the state of a run?", "user_id": "alice"}, "expected": {"must_cite": true, "cited_subset_of_read": true}, "tags": ["routine", "persistence", "smoke"], "source": "requirement"}


**Your turn:** add one case from a failure you wrote down in Notebook 1 (new unique `id`). The cell adds `S1-01` once.

In [3]:
mine = {"id": "S1-01", "input": {"question": "What is the tallest mountain in the world?", "user_id": "alice"},
        "expected": {"refuse": True, "must_not_call_tools": ["read_doc"]}, "tags": ["negative", "student"], "source": "failure"}
if mine["id"] not in {c["id"] for c in cases}:
    cases.append(mine)
    with open("../data/eval_cases.jsonl", "a") as f:
        f.write(json.dumps(mine) + "\n")

## 2. Upload the cases as a Langfuse dataset
A **dataset item** is an input with its expected output. Giving each item your own `id` (from the case id) makes the upload an update:
running the cell twice does not duplicate items. In the UI: **Datasets → research-brief-eval → Items**.

In [4]:
DATASET = "research-brief-eval"
try:
    langfuse.get_dataset(DATASET)
except Exception:
    langfuse.create_dataset(name=DATASET, description="Research Brief Agent eval cases")
for c in cases:
    langfuse.create_dataset_item(dataset_name=DATASET, id=f"{DATASET}--{c['id']}", input=c["input"],
                                 expected_output=c["expected"], metadata={"case_id": c["id"], "tags": c["tags"]})
langfuse.flush()
dataset = langfuse.get_dataset(DATASET)
print(DATASET, len(dataset.items), "items")

research-brief-eval 26 items


## 3. The task: the agent with a prompt version from Langfuse
`get_prompt("research-brief", label=...)` fetches each version created in the UI. The task runs the agent on one dataset item inside
`propagate_attributes(prompt=prompt)`, so every model call is linked to its prompt version. It returns what the evaluators read:
the answer, the tools that ran, the pages read, and `evidence`: those pages as one text (url, then the same text the agent saw).

In [5]:
def get_prompt(name, label="production"):
    """Fetch a prompt created in the Langfuse UI (README, step 5); stop with a clear message if it is missing."""
    try:
        return langfuse.get_prompt(name, label=label)
    except Exception as exc:
        raise SystemExit(f"Prompt {name!r} with label {label!r} not found in Langfuse: create it (README, step 5)") from exc

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, ToolMessage
from langfuse import propagate_attributes
from langfuse.langchain import CallbackHandler
from app.tools import TOOLS, UserContext

prompts = {label: get_prompt("research-brief", label=label) for label in ("v1", "v2")}
print({label: p.version for label, p in prompts.items()})
model = init_chat_model(os.environ["MODEL"])


def make_task(label):
    prompt = prompts[label]
    agent = create_agent(model, TOOLS, system_prompt=prompt.compile(), context_schema=UserContext)

    def task(*, item, **kwargs):
        question, user_id = item.input["question"], item.input["user_id"]
        with propagate_attributes(user_id=user_id, tags=[f"prompt-{label}"], prompt=prompt):
            messages = agent.invoke({"messages": [("user", question)]}, config={"callbacks": [CallbackHandler()]},
                                    context=UserContext(user_id))["messages"]
        results = {m.tool_call_id: m.text for m in messages if isinstance(m, ToolMessage)}
        calls = [c for m in messages if isinstance(m, AIMessage) for c in m.tool_calls if c["id"] in results]
        pages = {c["args"]["url"]: results[c["id"]] for c in calls
                 if c["name"] == "read_doc" and not results[c["id"]].startswith("ERROR")}
        evidence = "\n\n".join(f"[{url}]\n{text}" for url, text in pages.items())  # what the judge reads
        return {"answer": messages[-1].text, "tools": [c["name"] for c in calls], "pages": pages, "evidence": evidence}

    return task

{'v1': 1, 'v2': 2}


## 4. Evaluators
An evaluator gets the item's `input`, the task `output` and the item's `expected_output`, and returns an `Evaluation` (one **score**), or
`[]` when the case has no key for it.
- **Code checks** (same output, same score; fast and free): cites a url, cites only pages it read, calls the required tools, avoids the
  forbidden ones, declines out-of-scope questions.
- **LLM judge** (`groundedness`): one criterion (every claim is supported by a page the agent read), pass/fail, critique before the
  verdict. It sees the same page text as the agent and skips cases where the criterion does not apply (no page read, out-of-scope
  questions). Urls are not its job: `cited_subset_of_read` checks them in code. Its prompt `research-brief-judge` also lives in Langfuse;
  `JUDGE_MODEL` in `.env` can be a different model than the agent.
- **Run evaluator** `pass_rate`: one score for the whole run, the share of items that pass every code check (the judge is reported apart).

In [6]:
import re
from pydantic import BaseModel
from langfuse import Evaluation

URL = re.compile(r"https?://[^\s)\]>\"'`,]+")
REFUSAL = re.compile(r"(does not|doesn't|do not|not) (cover|answer|include)|not covered|outside|out of scope|"
                     r"(can't|cannot|unable to) (help|answer|provide)|no (relevant )?(information|documentation)", re.I)


def cited(answer):
    return {u.rstrip("./") for u in URL.findall(answer)}


def must_cite(*, output, expected_output, **kw):
    if not expected_output.get("must_cite"):
        return []
    n = len(cited(output["answer"]))
    return Evaluation(name="must_cite", value=float(n > 0), comment=f"{n} url(s) cited")


def cited_subset_of_read(*, output, expected_output, **kw):
    if not expected_output.get("cited_subset_of_read"):
        return []
    urls, read = cited(output["answer"]), {u.rstrip("/") for u in output["pages"]}
    return Evaluation(name="cited_subset_of_read", value=len(urls & read) / len(urls) if urls else 0.0,
                      comment=f"not read: {sorted(urls - read)}")


def must_call_tools(*, output, expected_output, **kw):
    if "must_call_tools" not in expected_output:
        return []
    ok = set(expected_output["must_call_tools"]) <= set(output["tools"])
    return Evaluation(name="must_call_tools", value=float(ok), comment=f"called {output['tools']}")


def must_not_call_tools(*, output, expected_output, **kw):
    if "must_not_call_tools" not in expected_output:
        return []
    bad = set(expected_output["must_not_call_tools"]) & set(output["tools"])
    return Evaluation(name="must_not_call_tools", value=float(not bad), comment=f"called {output['tools']}")


def refuse(*, output, expected_output, **kw):
    if not expected_output.get("refuse"):
        return []
    ok = bool(REFUSAL.search(output["answer"].replace("\u2019", "'")))
    return Evaluation(name="refuse", value=float(ok), comment=output["answer"][:120])


class Verdict(BaseModel):
    critique: str      # reasoning first, then the verdict
    passed: bool


judge_prompt = get_prompt("research-brief-judge")
judge = init_chat_model(os.getenv("JUDGE_MODEL") or os.environ["MODEL"]).with_structured_output(Verdict)


def groundedness(*, input, output, expected_output, **kw):
    if not output["pages"] or expected_output.get("refuse"):  # nothing to ground, or an out-of-scope case
        return []
    prompt = judge_prompt.compile(question=input["question"], brief=output["answer"], evidence=output["evidence"])
    v = judge.invoke(prompt)
    return Evaluation(name="groundedness", value=float(v.passed), comment=v.critique)


CODE_CHECKS = [must_cite, cited_subset_of_read, must_call_tools, must_not_call_tools, refuse]


def pass_rate(*, item_results, **kwargs):
    ok = [all(e.value == 1 for e in r.evaluations if e.name != "groundedness") for r in item_results]
    return Evaluation(name="pass_rate", value=sum(ok) / len(ok), comment=f"{sum(ok)}/{len(ok)} items pass every code check")

## 5. Experiment: prompt v1 vs prompt v2
An **experiment** runs the task on every item, applies the evaluators to each output and `pass_rate` to the run. Only the prompt changes;
items, model, tools and the cached pages stay the same. We run the 12 `smoke` items, three at a time; `data=` takes dataset items, so the
runs stay linked to the dataset.

In [7]:
logging.getLogger("app.tools").setLevel(logging.WARNING)  # quiet tool calls: one result line per run
smoke = [i for i in dataset.items if "smoke" in i.metadata["tags"]]
results = {}
for label in ("v1", "v2"):
    results[label] = langfuse.run_experiment(
        name=f"prompt-{label}", data=smoke, task=make_task(label),
        evaluators=CODE_CHECKS + [groundedness], run_evaluators=[pass_rate],
        max_concurrency=3, metadata={"prompt_label": label, "prompt_version": str(prompts[label].version)},
    )
    print(label, [(e.name, round(e.value, 2)) for e in results[label].run_evaluations], results[label].experiment_url)

v1 [('pass_rate', 0.75)] https://us.cloud.langfuse.com/project/cmumwadzp031sad0c12u9bfvq/experiments/results?baseline=205e1c217d92565a
v2 [('pass_rate', 1.0)] https://us.cloud.langfuse.com/project/cmumwadzp031sad0c12u9bfvq/experiments/results?baseline=5647430fd9c1781c


Open **Datasets → research-brief-eval → Experiments**, select both runs and click **Compare**: aggregate scores first, then the items that
got worse, then their traces. With one trial per case, one or two items of difference is noise; read the traces before you decide.
Read the critiques of the `groundedness` fails: each names the claim it could not find in the pages. A judge is calibrated against human
labels before you trust it (extension: annotation queue).

## 6. Scores per prompt version
**Prompts → research-brief → Metrics** shows, per version, latency, tokens, cost and the number of model calls linked to it, and the
**median score** of each score name. That score column only counts scores attached to a model call that is linked to the prompt, while the
experiment attaches its scores to the item's root span. The next cell copies each item's scores to the last linked model call of its trace
(it waits until the traces have arrived in Langfuse).

In [8]:
import time


def score_linked_generations(result, prompt_name="research-brief"):
    """Copy item scores to the last model call linked to the prompt, so Prompts → Metrics shows them per version."""
    langfuse.flush()
    pending = {r.trace_id: r.evaluations for r in result.item_results}
    for _ in range(60):                                # traces arrive a few seconds after flush()
        for trace_id in list(pending):
            gens = langfuse.api.observations.get_many(trace_id=trace_id, type="GENERATION", fields="core,prompt", limit=50).data
            linked = sorted((g for g in gens if g.prompt_name == prompt_name), key=lambda g: g.start_time)
            if linked:
                for e in pending.pop(trace_id):
                    langfuse.create_score(trace_id=trace_id, observation_id=linked[-1].id, name=e.name, value=e.value,
                                          comment=e.comment)
        if not pending:
            break
        time.sleep(5)
    langfuse.flush()


for result in results.values():
    score_linked_generations(result)
print("done: open Prompts → research-brief → Metrics")

done: open Prompts → research-brief → Metrics


### Promote the better version (in the UI)
v2 passes more cases, so it becomes `production`: open **Prompts → research-brief**, select version 2, **Labels → add `production`**.
Labels are unique across versions, so `production` leaves the older version. An app that calls `get_prompt("research-brief")` without a
label gets v2 within 60 seconds (the SDK cache), without a code change; rolling back means moving the label back.

## 7. Regression
A **regression case** is a failure you fixed and never want back; it is tagged `regression`. After every change of prompt, model or tool,
rerun these items and expect a pass rate near 100%.

In [9]:
regression = [i for i in dataset.items if "regression" in i.metadata["tags"]]
reg = langfuse.run_experiment(name="regression-v2", data=regression, task=make_task("v2"),
                              evaluators=CODE_CHECKS, run_evaluators=[pass_rate], max_concurrency=3)
for r in reg.item_results:
    failed = [f"{e.name}: {e.comment}" for e in r.evaluations if e.value < 1]
    print(r.item.metadata["case_id"], "FAIL " + "; ".join(failed) if failed else "pass")
print([(e.name, round(e.value, 2), e.comment) for e in reg.run_evaluations])

F5-01 pass
F3-01 pass
F2-02 pass
F2-01 pass
F1-01 pass
[('pass_rate', 1.0, '5/5 items pass every code check')]


### Extensions (optional)
- **Annotation queue:** in Langfuse, **Human Annotation**, create a queue with a boolean score config, add 5 traces, score them and
  compare with `groundedness`.
- **All cases:** use `data=dataset.items` instead of `smoke`.